In [3]:
# ============================================================
# WINE QUALITY NEURAL NETWORK EXPERIMENTS
# Activities 1, 2 and 3 + FINAL MODEL EVALUATION
#
# Activity 1:
#   Compare Sigmoid, Leaky ReLU and ELU
#
# Activity 2:
#   Optimise the optimiser for the three activations
#
# Activity 3:
#   Optimise neurons for activation + optimiser combinations
#
# IMPORTANT EVALUATION STRUCTURE
#
#   Training data
#        |
#        +----> Validation data
#        |
#        +----> Test data (UNTOUCHED)
#
# Activities 1-3 use VALIDATION data.
#
# The best configuration is then selected.
#
# Final model:
#   Train + Validation data
#              |
#              v
#        Final Test Evaluation
#
# The test set is used ONLY ONCE for the final evaluation.
#
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import gc
import time
import random

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 2. SETTINGS
# ============================================================

# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
#
# Use 10 initially if your laptop struggles.
#
# For the FINAL practical experiments, change to 50 if
# your practical requires 50 epochs.
#
# ------------------------------------------------------------

EPOCHS = 50

BATCH_SIZE = 32

RANDOM_STATE = 2


# ------------------------------------------------------------
# Result files
# ------------------------------------------------------------

if EPOCHS != 50:
    ACTIVITY_1_FILE = "activity1_results.csv"

    ACTIVITY_2_FILE = "activity2_results.csv"

    ACTIVITY_3_FILE = "activity3_results.csv"
else:     
    ACTIVITY_1_FILE = "activity1_results_50.csv"

    ACTIVITY_2_FILE = "activity2_results_50.csv"

    ACTIVITY_3_FILE = "activity3_results_50.csv"




# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

random.seed(RANDOM_STATE)

np.random.seed(RANDOM_STATE)

tf.random.set_seed(RANDOM_STATE)


# ============================================================
# 4. EXPERIMENT SETTINGS
# ============================================================

ACTIVATIONS = [
    "sigmoid",
    "leaky_relu",
    "elu"
]


OPTIMISERS = [
    "sgd",
    "rmsprop",
    "adagrad",
    "adadelta",
    "adam",
    "adamax",
    "nadam"
]


NEURONS = [
    8,
    16,
    32,
    64
]


# ============================================================
# 5. CHECK TENSORFLOW
# ============================================================

print("=" * 70)
print("TENSORFLOW INFORMATION")
print("=" * 70)

print("TensorFlow version:", tf.__version__)

print(
    "GPU devices detected:",
    tf.config.list_physical_devices("GPU")
)

print()


# ============================================================
# 6. LOAD DATASET
# ============================================================

print("=" * 70)
print("LOADING DATASET")
print("=" * 70)


if not os.path.exists("./wine.csv"):

    print("ERROR: wine.csv could not be found.")

    print()
    print("Current working directory:")
    print(os.getcwd())

    print()
    print("Files in current directory:")

    for file in os.listdir("."):
        print(file)

    raise FileNotFoundError(
        "wine.csv was not found in the current working directory."
    )


training_data = pd.read_csv("./wine.csv")


print("Dataset loaded successfully.")

print(
    "Dataset shape:",
    training_data.shape
)

print()

print("First five rows:")

print(
    training_data.head()
)

print()


# ============================================================
# 7. CONVERT QUALITY TO BINARY
# ============================================================

# good = 1
# bad  = 0

training_data["quality"] = training_data["quality"].replace({
    "good": 1,
    "bad": 0
})


# ------------------------------------------------------------
# Make sure quality is numeric
# ------------------------------------------------------------

training_data["quality"] = pd.to_numeric(
    training_data["quality"],
    errors="coerce"
)


# Remove rows with invalid target values

training_data = training_data.dropna(
    subset=["quality"]
)


# ============================================================
# 8. SEPARATE INPUTS AND TARGET
# ============================================================

X = training_data.drop(
    "quality",
    axis=1
)

y = training_data["quality"]


# ------------------------------------------------------------
# Convert input features to numeric
# ------------------------------------------------------------

X = X.apply(
    pd.to_numeric,
    errors="coerce"
)


# ============================================================
# 9. CHECK MISSING VALUES
# ============================================================

print("Missing values in X:")

print(
    X.isna().sum()
)

print()


# ------------------------------------------------------------
# Remove rows containing invalid input values
# ------------------------------------------------------------

valid_rows = ~X.isna().any(axis=1)

X = X.loc[valid_rows]

y = y.loc[valid_rows]


# ============================================================
# 10. CONVERT TO NUMPY
# ============================================================

X = X.to_numpy(
    dtype=np.float32
)

y = y.to_numpy(
    dtype=np.float32
)


print("X dtype:", X.dtype)

print("y dtype:", y.dtype)

print("Input shape:", X.shape)

print("Target shape:", y.shape)

print()


# ------------------------------------------------------------
# Automatically determine number of input features
# ------------------------------------------------------------

INPUT_FEATURES = X.shape[1]

print(
    "Number of input features:",
    INPUT_FEATURES
)

print()


# ============================================================
# 11. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# ------------------------------------------------------------
# IMPORTANT:
#
# The original script used:
#
#   70% training
#   30% testing
#
# We preserve the 70/30 split first.
#
# The 30% test set is then NEVER touched during Activities
# 1, 2 or 3.
#
# The 70% training portion is split again into:
#
#   80% training
#   20% validation
#
# Final proportions:
#
#   56% training
#   14% validation
#   30% testing
#
# ------------------------------------------------------------


train_val_x, test_x, train_val_y, test_y = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=y
)


train_x, validation_x, train_y, validation_y = train_test_split(
    train_val_x,
    train_val_y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=train_val_y
)


print("=" * 70)
print("TRAIN / VALIDATION / TEST SPLIT")
print("=" * 70)

print(
    "Training samples:",
    len(train_x)
)

print(
    "Validation samples:",
    len(validation_x)
)

print(
    "Testing samples:",
    len(test_x)
)

print()

print(
    "Training percentage:",
    f"{len(train_x) / len(X) * 100:.1f}%"
)

print(
    "Validation percentage:",
    f"{len(validation_x) / len(X) * 100:.1f}%"
)

print(
    "Testing percentage:",
    f"{len(test_x) / len(X) * 100:.1f}%"
)

print()


# ============================================================
# 12. CREATE MODEL
# ============================================================

def create_model(
    activation,
    optimiser,
    neurons
):

    model = keras.Sequential()

    # --------------------------------------------------------
    # Input layer
    # --------------------------------------------------------

    model.add(
        layers.Input(
            shape=(INPUT_FEATURES,)
        )
    )


    # --------------------------------------------------------
    # Hidden layer
    #
    # Leaky ReLU is created explicitly because it is a layer
    # rather than relying on a string activation.
    # --------------------------------------------------------

    if activation == "leaky_relu":

        model.add(
            layers.Dense(
                neurons
            )
        )

        model.add(
            layers.LeakyReLU(
                negative_slope=0.01
            )
        )

    else:

        model.add(
            layers.Dense(
                neurons,
                activation=activation
            )
        )


    # --------------------------------------------------------
    # Output layer
    # --------------------------------------------------------

    model.add(
        layers.Dense(
            1,
            activation="sigmoid"
        )
    )


    # ========================================================
    # Select optimiser
    # ========================================================

    if optimiser == "sgd":

        opt = keras.optimizers.SGD()

    elif optimiser == "rmsprop":

        opt = keras.optimizers.RMSprop()

    elif optimiser == "adagrad":

        opt = keras.optimizers.Adagrad()

    elif optimiser == "adadelta":

        opt = keras.optimizers.Adadelta()

    elif optimiser == "adam":

        opt = keras.optimizers.Adam()

    elif optimiser == "adamax":

        opt = keras.optimizers.Adamax()

    elif optimiser == "nadam":

        opt = keras.optimizers.Nadam()

    else:

        raise ValueError(
            f"Unknown optimiser: {optimiser}"
        )


    # ========================================================
    # Compile
    # ========================================================

    model.compile(
        loss="binary_crossentropy",
        optimizer=opt,
        metrics=["accuracy"]
    )


    return model


# ============================================================
# 13. EVALUATE ON VALIDATION DATA
# ============================================================

def evaluate_validation_model(model):

    predictions = model.predict(
        validation_x,
        batch_size=BATCH_SIZE,
        verbose=0
    )


    predictions = (
        predictions >= 0.5
    ).astype(int).flatten()


    accuracy = accuracy_score(
        validation_y,
        predictions
    )


    precision = precision_score(
        validation_y,
        predictions,
        zero_division=0
    )


    recall = recall_score(
        validation_y,
        predictions,
        zero_division=0
    )


    f1 = f1_score(
        validation_y,
        predictions,
        zero_division=0
    )


    return (
        accuracy,
        precision,
        recall,
        f1
    )


# ============================================================
# 14. CLEAN MEMORY
# ============================================================

def clean_memory():

    keras.backend.clear_session()

    gc.collect()


# ============================================================
# 15. ACTIVITY 1
# ============================================================

def run_activity_1():

    print()
    print("=" * 70)
    print("ACTIVITY 1")
    print("ACTIVATION FUNCTION COMPARISON")
    print("=" * 70)


    # --------------------------------------------------------
    # Load existing results
    # --------------------------------------------------------

    if os.path.exists(ACTIVITY_1_FILE):

        results = pd.read_csv(
            ACTIVITY_1_FILE
        )

        completed = set(
            results["activation"].astype(str)
        )

        print(
            "Existing results found:",
            len(results)
        )

    else:

        results = pd.DataFrame(
            columns=[
                "activation",
                "optimiser",
                "neurons",
                "epochs",
                "accuracy",
                "precision",
                "recall",
                "f1"
            ]
        )

        completed = set()

        print(
            "Starting Activity 1 from beginning."
        )


    # --------------------------------------------------------
    # Run experiments
    # --------------------------------------------------------

    for activation in ACTIVATIONS:

        if activation in completed:

            print(
                f"Skipping {activation} - already completed."
            )

            continue


        print()
        print("-" * 70)

        print(
            "Activation:",
            activation
        )

        print(
            "Optimiser: SGD"
        )

        print(
            "Neurons: 32"
        )

        print("-" * 70)


        start_time = time.time()


        try:

            # Reproducibility
            tf.random.set_seed(
                RANDOM_STATE
            )


            model = create_model(
                activation=activation,
                optimiser="sgd",
                neurons=32
            )


            model.fit(
                train_x,
                train_y,
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                verbose=0
            )


            (
                accuracy,
                precision,
                recall,
                f1
            ) = evaluate_validation_model(model)


            new_result = {
                "activation": activation,
                "optimiser": "sgd",
                "neurons": 32,
                "epochs": EPOCHS,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1
            }


            results = pd.concat(
                [
                    results,
                    pd.DataFrame([new_result])
                ],
                ignore_index=True
            )


            # Save immediately
            results.to_csv(
                ACTIVITY_1_FILE,
                index=False
            )


            elapsed = time.time() - start_time


            print()
            print(
                f"Validation Accuracy:  {accuracy:.4f}"
            )

            print(
                f"Validation Precision: {precision:.4f}"
            )

            print(
                f"Validation Recall:    {recall:.4f}"
            )

            print(
                f"Validation F1 Score:  {f1:.4f}"
            )

            print(
                f"Time:                 {elapsed:.1f} seconds"
            )

            print(
                f"Saved:                {ACTIVITY_1_FILE}"
            )


            del model

            clean_memory()


        except Exception as e:

            print()
            print("ERROR:")
            print(e)

            clean_memory()

            raise


    print()
    print("Activity 1 completed.")

    return results


# ============================================================
# 16. ACTIVITY 2
# ============================================================

def run_activity_2():

    print()
    print("=" * 70)
    print("ACTIVITY 2")
    print("ACTIVATION + OPTIMISER")
    print("=" * 70)


    if os.path.exists(ACTIVITY_2_FILE):

        results = pd.read_csv(
            ACTIVITY_2_FILE
        )

        completed = set(
            zip(
                results["activation"].astype(str),
                results["optimiser"].astype(str)
            )
        )

        print(
            "Existing results found:",
            len(results)
        )

    else:

        results = pd.DataFrame(
            columns=[
                "activation",
                "optimiser",
                "neurons",
                "epochs",
                "accuracy",
                "precision",
                "recall",
                "f1"
            ]
        )

        completed = set()

        print(
            "Starting Activity 2 from beginning."
        )


    # --------------------------------------------------------
    # Run experiments
    # --------------------------------------------------------

    for activation in ACTIVATIONS:

        for optimiser in OPTIMISERS:

            experiment = (
                activation,
                optimiser
            )


            if experiment in completed:

                print(
                    f"Skipping: "
                    f"{activation} + {optimiser}"
                )

                continue


            print()
            print("-" * 70)

            print(
                f"Activation: {activation}"
            )

            print(
                f"Optimiser:  {optimiser}"
            )

            print(
                "Neurons:    32"
            )

            print("-" * 70)


            start_time = time.time()


            try:

                tf.random.set_seed(
                    RANDOM_STATE
                )


                model = create_model(
                    activation=activation,
                    optimiser=optimiser,
                    neurons=32
                )


                model.fit(
                    train_x,
                    train_y,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    verbose=0
                )


                (
                    accuracy,
                    precision,
                    recall,
                    f1
                ) = evaluate_validation_model(model)


                new_result = {
                    "activation": activation,
                    "optimiser": optimiser,
                    "neurons": 32,
                    "epochs": EPOCHS,
                    "accuracy": accuracy,
                    "precision": precision,
                    "recall": recall,
                    "f1": f1
                }


                results = pd.concat(
                    [
                        results,
                        pd.DataFrame([new_result])
                    ],
                    ignore_index=True
                )


                results.to_csv(
                    ACTIVITY_2_FILE,
                    index=False
                )


                elapsed = time.time() - start_time


                print()
                print(
                    f"Validation Accuracy:  {accuracy:.4f}"
                )

                print(
                    f"Validation Precision: {precision:.4f}"
                )

                print(
                    f"Validation Recall:    {recall:.4f}"
                )

                print(
                    f"Validation F1 Score:  {f1:.4f}"
                )

                print(
                    f"Time:                 {elapsed:.1f} seconds"
                )

                print(
                    f"Saved:                {ACTIVITY_2_FILE}"
                )


                del model

                clean_memory()


            except Exception as e:

                print()
                print("ERROR:")
                print(e)

                clean_memory()

                raise


    print()
    print("Activity 2 completed.")

    return results


# ============================================================
# 17. ACTIVITY 3
# ============================================================

def run_activity_3():

    print()
    print("=" * 70)
    print("ACTIVITY 3")
    print("ACTIVATION + OPTIMISER + NEURONS")
    print("=" * 70)


    if os.path.exists(ACTIVITY_3_FILE):

        results = pd.read_csv(
            ACTIVITY_3_FILE
        )

        completed = set(
            zip(
                results["activation"].astype(str),
                results["optimiser"].astype(str),
                results["neurons"].astype(int)
            )
        )

        print(
            "Existing results found:",
            len(results)
        )

    else:

        results = pd.DataFrame(
            columns=[
                "activation",
                "optimiser",
                "neurons",
                "epochs",
                "accuracy",
                "precision",
                "recall",
                "f1"
            ]
        )

        completed = set()

        print(
            "Starting Activity 3 from beginning."
        )


    # --------------------------------------------------------
    # Run all combinations
    # --------------------------------------------------------

    for activation in ACTIVATIONS:

        for optimiser in OPTIMISERS:

            for neurons in NEURONS:

                experiment = (
                    activation,
                    optimiser,
                    neurons
                )


                if experiment in completed:

                    print(
                        f"Skipping: "
                        f"{activation} + "
                        f"{optimiser} + "
                        f"{neurons} neurons"
                    )

                    continue


                print()
                print("-" * 70)

                print(
                    f"Activation: {activation}"
                )

                print(
                    f"Optimiser:  {optimiser}"
                )

                print(
                    f"Neurons:    {neurons}"
                )

                print("-" * 70)


                start_time = time.time()


                try:

                    tf.random.set_seed(
                        RANDOM_STATE
                    )


                    model = create_model(
                        activation=activation,
                        optimiser=optimiser,
                        neurons=neurons
                    )


                    model.fit(
                        train_x,
                        train_y,
                        epochs=EPOCHS,
                        batch_size=BATCH_SIZE,
                        verbose=0
                    )


                    (
                        accuracy,
                        precision,
                        recall,
                        f1
                    ) = evaluate_validation_model(model)


                    new_result = {
                        "activation": activation,
                        "optimiser": optimiser,
                        "neurons": neurons,
                        "epochs": EPOCHS,
                        "accuracy": accuracy,
                        "precision": precision,
                        "recall": recall,
                        "f1": f1
                    }


                    results = pd.concat(
                        [
                            results,
                            pd.DataFrame([new_result])
                        ],
                        ignore_index=True
                    )


                    # Save immediately
                    results.to_csv(
                        ACTIVITY_3_FILE,
                        index=False
                    )


                    elapsed = time.time() - start_time


                    print()
                    print(
                        f"Validation Accuracy:  {accuracy:.4f}"
                    )

                    print(
                        f"Validation Precision: {precision:.4f}"
                    )

                    print(
                        f"Validation Recall:    {recall:.4f}"
                    )

                    print(
                        f"Validation F1 Score:  {f1:.4f}"
                    )

                    print(
                        f"Time:                 {elapsed:.1f} seconds"
                    )

                    print(
                        f"Saved:                {ACTIVITY_3_FILE}"
                    )


                    del model

                    clean_memory()


                except Exception as e:

                    print()
                    print("ERROR:")
                    print(e)

                    clean_memory()

                    raise


    print()
    print("Activity 3 completed.")

    return results


# ============================================================
# 18. RUN EXPERIMENTS
# ============================================================

print()
print("=" * 70)
print("READY TO START")
print("=" * 70)

print()

print(
    f"Epochs: {EPOCHS}"
)

print(
    f"Batch size: {BATCH_SIZE}"
)

print()

print(
    "Activities 1-3 evaluate on VALIDATION data."
)

print(
    "The TEST data will remain untouched."
)

print()


# ------------------------------------------------------------
# IMPORTANT
#
# Run these individually if you want to monitor laptop load.
# ------------------------------------------------------------

# Activity 1 = 3 experiments
# Activity 2 = 21 experiments
# Activity 3 = 84 experiments


# ------------------------------------------------------------
# Activity 1
# ------------------------------------------------------------

results1 = run_activity_1()


# ------------------------------------------------------------
# Activity 2
# ------------------------------------------------------------

results2 = run_activity_2()


# ------------------------------------------------------------
# Activity 3
# ------------------------------------------------------------

results3 = run_activity_3()


# ============================================================
# 19. DISPLAY ACTIVITY 1 RESULTS
# ============================================================

results1 = pd.read_csv(
    ACTIVITY_1_FILE
)


print()
print("=" * 70)
print("ACTIVITY 1 RESULTS - VALIDATION F1")
print("=" * 70)


print(
    results1.sort_values(
        by="f1",
        ascending=False
    ).to_string(index=False)
)


# ============================================================
# 20. DISPLAY ACTIVITY 2 RESULTS
# ============================================================

results2 = pd.read_csv(
    ACTIVITY_2_FILE
)


print()
print("=" * 70)
print("ACTIVITY 2 RESULTS - VALIDATION F1")
print("=" * 70)


print(
    results2.sort_values(
        by="f1",
        ascending=False
    ).to_string(index=False)
)


print()
print("=" * 70)
print("ACTIVITY 2 - AVERAGE RESULTS BY ACTIVATION")
print("=" * 70)


print(
    results2.groupby("activation")[
        [
            "accuracy",
            "precision",
            "recall",
            "f1"
        ]
    ].mean()
)


# ============================================================
# 21. DISPLAY ACTIVITY 3 RESULTS
# ============================================================

results3 = pd.read_csv(
    ACTIVITY_3_FILE
)


print()
print("=" * 70)
print("ACTIVITY 3 - TOP 20 BY VALIDATION F1")
print("=" * 70)


print(
    results3.sort_values(
        by="f1",
        ascending=False
    ).head(20).to_string(index=False)
)


print()
print("=" * 70)
print("ACTIVITY 3 - TOP 20 BY VALIDATION ACCURACY")
print("=" * 70)


print(
    results3.sort_values(
        by="accuracy",
        ascending=False
    ).head(20).to_string(index=False)
)


# ============================================================
# 22. SELECT BEST CONFIGURATION
# ============================================================

print()
print("=" * 70)
print("SELECTING FINAL MODEL CONFIGURATION")
print("=" * 70)


# ------------------------------------------------------------
# F1 is used as the selection metric because this is the metric
# used throughout the experiment comparison.
#
# IMPORTANT:
# This is VALIDATION F1, NOT TEST F1.
# ------------------------------------------------------------

best_index = results3["f1"].idxmax()

best_result = results3.loc[
    best_index
]


BEST_ACTIVATION = str(
    best_result["activation"]
)

BEST_OPTIMISER = str(
    best_result["optimiser"]
)

BEST_NEURONS = int(
    best_result["neurons"]
)


print()

print(
    "Selected configuration:"
)

print(
    "Activation:",
    BEST_ACTIVATION
)

print(
    "Optimiser:",
    BEST_OPTIMISER
)

print(
    "Neurons:",
    BEST_NEURONS
)

print()

print(
    "Validation Accuracy:",
    f"{best_result['accuracy']:.4f}"
)

print(
    "Validation Precision:",
    f"{best_result['precision']:.4f}"
)

print(
    "Validation Recall:",
    f"{best_result['recall']:.4f}"
)

print(
    "Validation F1:",
    f"{best_result['f1']:.4f}"
)


# ============================================================
# 23. FINAL MODEL
# ============================================================

def evaluate_final_model(
    activation,
    optimiser,
    neurons
):

    print()
    print("=" * 70)
    print("FINAL MODEL EVALUATION")
    print("=" * 70)

    print()
    print(
        "Final configuration:"
    )

    print(
        f"Activation: {activation}"
    )

    print(
        f"Optimiser:  {optimiser}"
    )

    print(
        f"Neurons:    {neurons}"
    )

    print()

    print(
        "Training final model on:"
    )

    print(
        f"Training + Validation samples: "
        f"{len(train_val_x)}"
    )

    print(
        f"Final test samples: "
        f"{len(test_x)}"
    )

    print()


    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    tf.random.set_seed(
        RANDOM_STATE
    )


    # --------------------------------------------------------
    # Create final model
    # --------------------------------------------------------

    model = create_model(
        activation=activation,
        optimiser=optimiser,
        neurons=neurons
    )


    # --------------------------------------------------------
    # IMPORTANT:
    #
    # Now that the configuration has been selected using the
    # validation data, combine training + validation data.
    #
    # The test set remains completely untouched.
    # --------------------------------------------------------

    final_train_x = np.concatenate(
        [
            train_x,
            validation_x
        ],
        axis=0
    )


    final_train_y = np.concatenate(
        [
            train_y,
            validation_y
        ],
        axis=0
    )


    print(
        "Final training data shape:",
        final_train_x.shape
    )


    # --------------------------------------------------------
    # Train final model
    # --------------------------------------------------------

    model.fit(
        final_train_x,
        final_train_y,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0
    )


    # ========================================================
    # FINAL TEST PREDICTIONS
    # ========================================================

    probabilities = model.predict(
        test_x,
        batch_size=BATCH_SIZE,
        verbose=0
    )


    predictions = (
        probabilities >= 0.5
    ).astype(int).flatten()


    # ========================================================
    # FINAL TEST METRICS
    # ========================================================

    accuracy = accuracy_score(
        test_y,
        predictions
    )


    precision = precision_score(
        test_y,
        predictions,
        zero_division=0
    )


    recall = recall_score(
        test_y,
        predictions,
        zero_division=0
    )


    f1 = f1_score(
        test_y,
        predictions,
        zero_division=0
    )


    print()
    print("=" * 70)
    print("FINAL TEST RESULTS")
    print("=" * 70)

    print()

    print(
        f"Accuracy:  {accuracy:.4f}"
    )

    print(
        f"Precision: {precision:.4f}"
    )

    print(
        f"Recall:    {recall:.4f}"
    )

    print(
        f"F1 Score:  {f1:.4f}"
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    print()
    print("=" * 70)
    print("FINAL TEST CONFUSION MATRIX")
    print("=" * 70)

    cm = confusion_matrix(
        test_y,
        predictions
    )


    print()

    print(cm)


    # --------------------------------------------------------
    # Make the matrix easier to interpret
    # --------------------------------------------------------

    print()

    print(
        "Confusion Matrix Layout:"
    )

    print(
        "[[True Negative, False Positive],"
    )

    print(
        " [False Negative, True Positive]]"
    )


    # ========================================================
    # CLASSIFICATION REPORT
    # ========================================================

    print()
    print("=" * 70)
    print("FINAL TEST CLASSIFICATION REPORT")
    print("=" * 70)

    print()

    print(
        classification_report(
            test_y,
            predictions,
            target_names=[
                "Bad (0)",
                "Good (1)"
            ],
            zero_division=0
        )
    )


    # ========================================================
    # RETURN RESULTS
    # ========================================================

    final_results = {
        "activation": activation,
        "optimiser": optimiser,
        "neurons": neurons,
        "epochs": EPOCHS,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


    # --------------------------------------------------------
    # Save final results
    # --------------------------------------------------------

    final_results_df = pd.DataFrame(
        [final_results]
    )


    final_results_df.to_csv(
        "final_model_results.csv",
        index=False
    )


    print(
        "Final results saved to:",
        "final_model_results.csv"
    )


    # --------------------------------------------------------
    # Clean memory
    # --------------------------------------------------------

    del model

    clean_memory()


    return final_results


# ============================================================
# 24. RUN FINAL MODEL
# ============================================================

final_results = evaluate_final_model(
    activation=BEST_ACTIVATION,
    optimiser=BEST_OPTIMISER,
    neurons=BEST_NEURONS
)


# ============================================================
# 25. FINAL SUMMARY
# ============================================================

print()
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print()

print(
    "Selected configuration:"
)

print(
    f"Activation: {final_results['activation']}"
)

print(
    f"Optimiser:  {final_results['optimiser']}"
)

print(
    f"Neurons:    {final_results['neurons']}"
)

print()

print(
    "FINAL TEST RESULTS:"
)

print(
    f"Accuracy:  {final_results['accuracy']:.4f}"
)

print(
    f"Precision: {final_results['precision']:.4f}"
)

print(
    f"Recall:    {final_results['recall']:.4f}"
)

print(
    f"F1 Score:  {final_results['f1']:.4f}"
)

print()

print("=" * 70)
print("EXPERIMENTS AND FINAL EVALUATION COMPLETE")
print("=" * 70)

TENSORFLOW INFORMATION
TensorFlow version: 2.21.0
GPU devices detected: []

LOADING DATASET
Dataset loaded successfully.
Dataset shape: (1599, 12)

First five rows:
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58